## Ridge Regression

### sklearn

In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [43]:
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge

In [44]:
df = pd.read_csv('advertising.csv')
df.head()

,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,12.0
3,151.5,41.3,58.5,16.5
4,180.8,10.8,58.4,17.9


In [45]:
# features and target
x = df.iloc[:, 0:3]
y = df.iloc[:, -1]

x = (x - x.mean()) / x.std()
y = (y - y.mean()) / y.std()

In [46]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=4)

In [47]:
r = Ridge(alpha=2)
r.fit(x_train, y_train)
y_pred = r.predict(x_test)

print('intercept: ',r.intercept_)
print('coef: ',r.coef_)
print('r2 Score: ',r2_score(y_test, y_pred))

intercept:  -0.008328586402503182
coef:  [0.8816195  0.28651265 0.006791  ]
r2 Score:  0.9049943116203563


### from Scratch (ridge)

In [48]:
class ridge:
    def __init__(self, alpha):
        self.coef_ = None
        self.intercept_ = None
        self.alpha = alpha
    
    def fit(self, x_train, x_test):
        x_train = np.insert(x_train, 0, 1, axis = 1)
        id = np.identity(x_train.shape[1])
        id[0,0] = 0

        p = x_train.T @ x_train
        p_inv = np.linalg.inv(p)
        q = x_train.T @ y_train

        betas = np.dot(p_inv, q)
        self.intercept_ = betas[0]
        self.coef_ = betas[1: ]

        print('Intercept = ',self.intercept_)
        print('coef = ',self.coef_)

    def predict(self, x_test):
        return self.intercept_ + np.dot(x_test, self.coef_)

In [49]:
ridge_obj = ridge(2)
ridge_obj.fit(x_train, y_train)
y_pred_obj = ridge_obj.predict(x_test)

print('R2 score: ',r2_score(y_test, y_pred_obj))

Intercept =  -0.007423526624769229
coef =  [0.89242352 0.29027842 0.00475238]
R2 score:  0.9046414171169502


### Gradient Descent from Scratch (ridge)

In [58]:
class RidgeGD:
    def __init__(self, alpha, learning_rate, epochs):
        self.alpha = alpha
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.intercept_ = None
        self.coef_ = None
    
    def fit(self, x_train, y_train):
        m = x_train.shape[0]
        self.coef_ = np.zeros(x_train.shape[1]) # Start at 0 for better stability
        self.intercept_ = 0
        
        X_ext = np.insert(x_train, 0, 1, axis=1)
        thetas = np.insert(self.coef_, 0, self.intercept_)

        for i in range(self.epochs):
            # 1. Regularization term (don't penalize index 0)
            reg_term = np.copy(thetas)
            reg_term[0] = 0
            
            # 2. Gradient with (1/m) scaling
            # Correct MSE Gradient: (1/m) * (X^T @ (X @ theta - y)) + (alpha/m) * theta
            # Note: We divide alpha by m to keep the penalty proportional to data size
            grad = (1/m) * ((X_ext.T @ X_ext @ thetas) - (X_ext.T @ y_train) + (self.alpha * reg_term))
            
            thetas = thetas - (self.learning_rate * grad)

        self.intercept_ = thetas[0]
        self.coef_ = thetas[1:]
        print('intercept:', self.intercept_)
        print('coef:', self.coef_)
        
    
    def predict(self,x_test):
        return self.intercept_ + np.dot(x_test, self.coef_)

In [59]:
gd = RidgeGD(2, 0.1, 100)
gd.fit(x_train, y_train)
y_pred = gd.predict(x_test)

print('R2 score: ',r2_score(y_test, y_pred))

intercept: -0.008385100686661708
coef: [0.88158681 0.28630192 0.00699421]
R2 score:  0.9049632169166956
